# HTML Tag Autocomplete with a Character-Level LSTM

This notebook trains a tiny character-level LSTM that learns to predict the next character of simple HTML tags such as `<input type='text'>`. Given a starting string, it repeatedly predicts the next character until it outputs `>`. It is a toy example of how sequence models power text autocomplete.

**How to run:** Open in Google Colab (CPU is fine) and run all cells.

_Rewritten from my original Colab notebook: `Untitled2.ipynb`._

## 1. Imports and seeds

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

np.random.seed(42)
tf.random.set_seed(42)

## 2. Dataset

Each tag ends with `>` so the model can learn when a tag is finished. Index 0 is reserved for padding.

In [ ]:
html_data = ["<form>", "<input type='text'>", "<input type='password'>", "<input type='submit'>",
             "<input type='checkbox'>", "<input type='radio'>", "<input type='email'>", "<input type='button'>"]

chars = sorted(set("".join(html_data)))
char_to_int = {c: i + 1 for i, c in enumerate(chars)}  # 0 = padding
int_to_char = {i: c for c, i in char_to_int.items()}
vocab_size = len(chars) + 1

X_seqs, y_chars = [], []
for tag in html_data:
    for i in range(1, len(tag)):
        X_seqs.append([char_to_int[c] for c in tag[:i]])  # prefix
        y_chars.append(char_to_int[tag[i]])                # next character

max_len = max(len(s) for s in X_seqs)
X = pad_sequences(X_seqs, maxlen=max_len, padding="pre")
y = np.array(y_chars)
print("Training examples:", X.shape, "vocab size:", vocab_size)

## 3. Model and training

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(vocab_size, 32),
    layers.LSTM(128),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X, y, epochs=200, batch_size=32, verbose=0)
print("Final training accuracy:", round(history.history["accuracy"][-1], 3))

## 4. Autocomplete

The model only knows characters from the training data; unknown characters are skipped.

In [ ]:
def complete_html(prefix, max_new_chars=30):
    result = prefix
    for _ in range(max_new_chars):
        encoded = [char_to_int[c] for c in result if c in char_to_int]
        encoded = pad_sequences([encoded], maxlen=max_len, padding="pre")
        next_index = int(np.argmax(model.predict(encoded, verbose=0)[0]))
        next_char = int_to_char.get(next_index, "")  # index 0 (padding) -> stop
        result += next_char
        if next_char in (">", ""):
            break
    return result

for start in ["<f", "<input type='p", "<input type='c", "<input type='e"]:
    print(f"{start!r:20} -> {complete_html(start)}")